<a href="https://colab.research.google.com/github/Ali-Hamza-developer/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review
**Lane: Refresh / Content Opportunity Scoring**

Run top to bottom (Runtime → Run all). Requires `HF_TOKEN` Colab Secret, same as w03/w05.

In [2]:
!pip install -q duckdb

import duckdb
import pandas as pd
import os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
os.makedirs('work/outputs', exist_ok=True)
print('DuckDB ready, target month =', MONTH)

DuckDB ready, target month = 2026-03


In [3]:
# Rebuild the same feature base as w03/w05 — real columns, confirmed schema
base = con.sql(f"""
    WITH window_90d AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_90d,
            SUM(gsc_clicks) AS clicks_90d,
            AVG(gsc_avg_position) AS avg_position_90d
        FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        w.client_hash_id,
        w.content_hash_id,
        w.impressions_90d,
        w.clicks_90d,
        w.avg_position_90d,
        CASE WHEN w.impressions_90d > 0 THEN w.clicks_90d::DOUBLE / w.impressions_90d ELSE NULL END AS ctr_90d,
        (DATE '2026-03-31' - COALESCE(d.last_optimized_date, d.content_created_date)) AS days_since_last_update,
        d.word_count
    FROM window_90d w
    JOIN read_parquet('{REL}/dim_content.parquet') d
      ON w.content_hash_id = d.content_hash_id AND w.client_hash_id = d.client_hash_id
    WHERE w.impressions_90d > 0
      AND (DATE '2026-03-31' - COALESCE(d.last_optimized_date, d.content_created_date)) >= 0
""").df()
print('rows:', len(base))
base.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 136974


,client_hash_id,content_hash_id,impressions_90d,clicks_90d,avg_position_90d,ctr_90d,days_since_last_update,word_count
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,0.000000,47,3579
1,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,6.015432,0.000000,47,3249
2,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,5.956862,0.001418,47,3067
3,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,12.977513,0.000000,47,3344
4,client_62f4a7e64f5e0096,content_df22bda1218f13ff,2099.0,1.0,3.066796,0.000476,47,3698


## 1. My rule and its reason codes

**Rule in plain words:** flag a page for review when it has real search visibility (enough impressions to matter) AND at least one of: (a) it under-captures clicks for its position tier, or (b) it hasn't been touched in a long time. Position and CTR are checked together because comparing CTR across different positions without adjusting for position is a named mistake in the lane guide — a page at position 25 is *expected* to have low CTR, that's not a signal, so we only flag low CTR for pages that are actually visible (top ~20 positions).

**Signal check 1 — position vs. CTR** (flag-linked: this is the signal behind FlyRank's real CTR-fix logic)

**Signal check 2 — staleness vs. future outcome** (flag-linked: this is the signal behind the refresh flags)

**Reason codes this rule can output:**
- `low_ctr_visible_page` — good position, impressions clear the floor, CTR below expected
- `stale_visible_page` — long time since last update, impressions clear the floor
- `low_ctr_and_stale` — both conditions true
- `monitor_only` — visible but neither condition triggers

In [4]:
# Signal check 1: position vs CTR, bucketed by position tier (never compare CTR across tiers directly)
signal1 = base.copy()
signal1['position_tier'] = pd.cut(
    signal1['avg_position_90d'],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=['1-3', '4-10', '11-20', '21-50', '50+']
)

signal1_table = signal1.groupby('position_tier', observed=True).agg(
    n=('ctr_90d', 'count'),
    mean_ctr=('ctr_90d', 'mean'),
    median_ctr=('ctr_90d', 'median')
).reset_index()

print(signal1_table.to_string())
print()
print('VERDICT: fill in after reading the table — CONFIRMED if mean/median CTR clearly')
print('decreases as position tier worsens (1-3 highest, 50+ lowest); OPPOSITE if reversed;')
print('MIXED if no clear monotonic pattern; FALSE if tiers show no meaningful difference.')

  position_tier      n  mean_ctr  median_ctr
0           1-3  13407  0.011890         0.0
1          4-10  60221  0.005416         0.0
2         11-20  23243  0.003263         0.0
3         21-50  27182  0.002381         0.0
4           50+  11510  0.000905         0.0

VERDICT: fill in after reading the table — CONFIRMED if mean/median CTR clearly
decreases as position tier worsens (1-3 highest, 50+ lowest); OPPOSITE if reversed;
MIXED if no clear monotonic pattern; FALSE if tiers show no meaningful difference.


In [5]:
# Signal check 2: staleness vs actual future outcome (30-day forward window, month=2026-04)
label_window = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_next30
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    WHERE report_date <= DATE '2026-04-30'
    GROUP BY 1, 2
""").df()

signal2 = base.merge(label_window, on=['client_hash_id', 'content_hash_id'], how='inner')
signal2['future_decline'] = (signal2['impressions_next30'] < 0.7 * signal2['impressions_90d']).astype(int)
signal2['staleness_bucket'] = pd.cut(
    signal2['days_since_last_update'],
    bins=[-1, 30, 90, 180, 365, 100000],
    labels=['0-30d', '31-90d', '91-180d', '181-365d', '365d+']
)

signal2_table = signal2.groupby('staleness_bucket', observed=True).agg(
    n=('future_decline', 'count'),
    decline_rate=('future_decline', 'mean')
).reset_index()

print(signal2_table.to_string())
print()
print('VERDICT: fill in after reading the table — CONFIRMED if decline_rate clearly rises with')
print('staleness; OPPOSITE if it falls; MIXED if no clear trend; FALSE if buckets look the same.')
print('A clean FALSE/OPPOSITE here is a real, useful result — it means staleness alone should')
print('NOT drive the rule and you should say so honestly in your write-up.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  staleness_bucket      n  decline_rate
0            0-30d  14529      0.245784
1           31-90d  32220      0.496772
2          91-180d  20159      0.530383
3         181-365d  53992      0.591828
4            365d+  16073      0.507746

VERDICT: fill in after reading the table — CONFIRMED if decline_rate clearly rises with
staleness; OPPOSITE if it falls; MIXED if no clear trend; FALSE if buckets look the same.
A clean FALSE/OPPOSITE here is a real, useful result — it means staleness alone should
NOT drive the rule and you should say so honestly in your write-up.


**Write your two verdicts here** (fill in after reading the two tables above, with the `n` values quoted):
- Position vs CTR: **[CONFIRMED / OPPOSITE / MIXED / FALSE]** — n=..., one sentence on what the table shows
- Staleness vs future decline: **[CONFIRMED / OPPOSITE / MIXED / FALSE]** — n=..., one sentence on what the table shows

## 2. Build the ranked queue (writes the CSV)

Score uses only prior-90-day signals — no future window, no product flags. Adjust the CTR floor / staleness cutoff below once you've read your Part 1 verdicts; if staleness came back FALSE or OPPOSITE, drop that half of the rule and say why in Part 4.

In [7]:
queue = base.copy()

# expected CTR floor by position tier — adjust using signal1_table's real mean_ctr values
queue['position_tier'] = pd.cut(
    queue['avg_position_90d'], bins=[0, 3, 10, 20, 50, 1000],
    labels=['1-3', '4-10', '11-20', '21-50', '50+']
)
tier_expected_ctr = signal1_table.set_index('position_tier')['mean_ctr'].to_dict()
queue['expected_ctr'] = queue['position_tier'].map(tier_expected_ctr).astype(float)

is_visible = queue['impressions_90d'] >= 250          # volume floor, adjust if too strict/loose
low_ctr = is_visible & (queue['avg_position_90d'] <= 20) & (queue['ctr_90d'] < 0.7 * queue['expected_ctr'])
is_stale = is_visible & (queue['days_since_last_update'] >= 180)

queue['reason_code'] = 'monitor_only'
queue.loc[low_ctr & ~is_stale, 'reason_code'] = 'low_ctr_visible_page'
queue.loc[is_stale & ~low_ctr, 'reason_code'] = 'stale_visible_page'
queue.loc[low_ctr & is_stale, 'reason_code'] = 'low_ctr_and_stale'

queue['action_label'] = queue['reason_code'].map({
    'low_ctr_visible_page': 'rewrite_title_meta',
    'stale_visible_page': 'refresh_content',
    'low_ctr_and_stale': 'refresh_and_rewrite',
    'monitor_only': 'monitor',
})

queue['score'] = (
    0.5 * low_ctr.astype(int) +
    0.3 * is_stale.astype(int) +
    0.2 * is_visible.astype(int)
)

ranked_queue = queue.sort_values('score', ascending=False).reset_index(drop=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print('wrote', len(ranked_queue), 'rows to work/outputs/baseline_action_score.csv')
ranked_queue[['client_hash_id','content_hash_id','score','reason_code','action_label']].head(20)

wrote 136974 rows to work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,score,reason_code,action_label
0,client_62f4a7e64f5e0096,content_ee4b8d1969c043e0,1.0,low_ctr_and_stale,refresh_and_rewrite
1,client_62f4a7e64f5e0096,content_f638ae3082741a75,1.0,low_ctr_and_stale,refresh_and_rewrite
2,client_62f4a7e64f5e0096,content_097c01176d33bad3,1.0,low_ctr_and_stale,refresh_and_rewrite
3,client_62f4a7e64f5e0096,content_46b738bc35bc46f8,1.0,low_ctr_and_stale,refresh_and_rewrite
4,client_62f4a7e64f5e0096,content_4f637680dc1f1f08,1.0,low_ctr_and_stale,refresh_and_rewrite
5,client_fef1a8f436438636,content_09232048711c5dbc,1.0,low_ctr_and_stale,refresh_and_rewrite
6,client_fef1a8f436438636,content_b53109ee359cb52a,1.0,low_ctr_and_stale,refresh_and_rewrite
7,client_fef1a8f436438636,content_c256d0300a0b4d0e,1.0,low_ctr_and_stale,refresh_and_rewrite
8,client_fef1a8f436438636,content_f8a879b77d7e8d0e,1.0,low_ctr_and_stale,refresh_and_rewrite
9,client_fef1a8f436438636,content_6a4b62a0e50cd582,1.0,low_ctr_and_stale,refresh_and_rewrite


## 3. Top-20 review

Fill this table using the **actual top 20 rows printed above**. For each: the action, why it's there (from the real feature values you can see in the full row), and what would make it wrong.

To see the full row detail per item, run the cell below and read it before writing each line — don't guess.

In [8]:
cols_to_review = ['client_hash_id','content_hash_id','impressions_90d','clicks_90d',
                   'avg_position_90d','ctr_90d','days_since_last_update','word_count',
                   'score','reason_code','action_label']
ranked_queue[cols_to_review].head(20)

,client_hash_id,content_hash_id,impressions_90d,clicks_90d,avg_position_90d,ctr_90d,days_since_last_update,word_count,score,reason_code,action_label
0,client_62f4a7e64f5e0096,content_ee4b8d1969c043e0,2931.0,11.0,2.769686,0.003753,266,<NA>,1.0,low_ctr_and_stale,refresh_and_rewrite
1,client_62f4a7e64f5e0096,content_f638ae3082741a75,1867.0,4.0,4.922981,0.002142,266,<NA>,1.0,low_ctr_and_stale,refresh_and_rewrite
2,client_62f4a7e64f5e0096,content_097c01176d33bad3,469.0,2.0,2.068731,0.004264,266,<NA>,1.0,low_ctr_and_stale,refresh_and_rewrite
3,client_62f4a7e64f5e0096,content_46b738bc35bc46f8,250.0,0.0,11.778646,0.000000,266,<NA>,1.0,low_ctr_and_stale,refresh_and_rewrite
4,client_62f4a7e64f5e0096,content_4f637680dc1f1f08,278.0,0.0,9.074465,0.000000,266,<NA>,1.0,low_ctr_and_stale,refresh_and_rewrite
5,client_fef1a8f436438636,content_09232048711c5dbc,626.0,1.0,15.770756,0.001597,214,1508,1.0,low_ctr_and_stale,refresh_and_rewrite
6,client_fef1a8f436438636,content_b53109ee359cb52a,1703.0,12.0,2.700924,0.007046,214,1199,1.0,low_ctr_and_stale,refresh_and_rewrite
7,client_fef1a8f436438636,content_c256d0300a0b4d0e,753.0,2.0,7.145853,0.002656,214,1528,1.0,low_ctr_and_stale,refresh_and_rewrite
8,client_fef1a8f436438636,content_f8a879b77d7e8d0e,674.0,0.0,12.282889,0.000000,214,1452,1.0,low_ctr_and_stale,refresh_and_rewrite
9,client_fef1a8f436438636,content_6a4b62a0e50cd582,381.0,0.0,1.787472,0.000000,214,1361,1.0,low_ctr_and_stale,refresh_and_rewrite


## 4. Weak picks + leakage check

**Weak picks:** from your top-20 review above, which ones look wrong on a second look, and why? (Fill in after step 3 — look for rows where the reason code technically triggered but the underlying numbers are borderline or noisy, e.g. `impressions_90d` barely above the floor.)

**Leakage check — confirm explicitly:**
- [ ] `score`, `reason_code`, `action_label` were built only from `base` (90-day prior window) and `signal1_table` (also prior-window). No column from `label_window` or `signal2` (which use `month=2026-04`, the forward window) touches the queue-building cell.
- [ ] No rebuilt product flag (`health_score`, `priority_score`, `action_type`) was used as an input to `score`.
- [ ] Verify by rerunning this check:

In [9]:
queue_inputs = ['impressions_90d','clicks_90d','avg_position_90d','ctr_90d','days_since_last_update','expected_ctr']
print('Columns used to build the queue score:', queue_inputs)
print('All derived from month=2026-03 only:', all('90d' in c or c in ['days_since_last_update','expected_ctr'] for c in queue_inputs))
print('label_window / future_decline / signal2 NOT referenced in the queue-building cell above (manual code review).')

Columns used to build the queue score: ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'ctr_90d', 'days_since_last_update', 'expected_ctr']
All derived from month=2026-03 only: True
label_window / future_decline / signal2 NOT referenced in the queue-building cell above (manual code review).


## 5. Self-check

- [ ] Every section filled — markdown thinking AND the code that backs it
- [ ] Notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to repo under `work/notebooks/w04_baseline_score.ipynb` — then submit repo URL on the card
- [ ] `work/outputs/baseline_action_score.csv` regenerates on run, stays out of git (CI leak-guard)